In [1]:
import networkx as nx


In [3]:
import pandas as pd

features = pd.read_csv("datasets/elliptic_txs_features.csv", header=None)
edges = pd.read_csv("datasets/elliptic_txs_edgelist.csv")
labels = pd.read_csv("datasets/elliptic_txs_classes.csv")


In [4]:
G_elliptic = nx.from_pandas_edgelist(edges, source='txId1', target='txId2')

elliptic_graph_stats = {
    "Nº Nós": G_elliptic.number_of_nodes(),
    "Nº Arestas": G_elliptic.number_of_edges(),
    "Grau Médio": sum(dict(G_elliptic.degree()).values()) / G_elliptic.number_of_nodes()
}

In [5]:
eth = pd.read_csv("datasets/transaction_dataset.csv")


In [6]:
G_eth = nx.from_pandas_edgelist(
    eth,
    source='from_address',
    target='to_address'
)

eth_graph_stats = {
    "Nº Nós": G_eth.number_of_nodes(),
    "Nº Arestas": G_eth.number_of_edges(),
    "Grau Médio": sum(dict(G_eth.degree()).values()) / G_eth.number_of_nodes()
}

KeyError: 'from_address'

In [ ]:
tabela_2 = pd.DataFrame.from_dict(
    {
        "Elliptic": elliptic_graph_stats,
        "Ethereum": eth_graph_stats
    },
    orient="index"
)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

In [ ]:
X = eth.drop(columns=['is_fraud'])
y = eth['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

tabular_results = {
    "AUC": roc_auc_score(y_test, y_prob),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred)
}

In [ ]:
tabela_3 = pd.DataFrame(tabular_results, index=["Regressão Logística"])

In [ ]:
import torch
from torch_geometric.data import Data

In [ ]:
edge_index = torch.tensor(list(G_eth.edges)).t().contiguous()
x = torch.tensor(X.values, dtype=torch.float)
y = torch.tensor(y.values, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv


In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 64)
        self.conv2 = GCNConv(64, 2)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

In [ ]:
model = GCN(data.num_node_features)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(30):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out, data.y)
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()
out = model(data)
pred = out.argmax(dim=1)

gnn_results = {
    "AUC": roc_auc_score(y.numpy(), out[:,1].detach().numpy()),
    "Precision": precision_score(y, pred),
    "Recall": recall_score(y, pred),
    "F1": f1_score(y, pred)
}